# Multi-Facility Modified vs Plan Profile Builder

CSV-only notebook for JupyterLite. The facility map uses column 1 for well name and column 2 for facility. The four large ComboCurve exports use A=Well Name, H=Date, I=Oil BBL/month, J=Gas MCF/month.

For each facility, the notebook creates one oil chart and one gas chart. Each chart overlays Modified and Plan rate profiles with Modified and Plan cumulative volumes, shades the cumulative delta red, annotates the final cumulative delta and percent versus Plan, and notes peak rates.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

PROFILE_MONTHS = 600
SAVE_PLOTS = True

# ============================================================
# EDIT THESE FILENAMES TO MATCH YOUR CSV FILES
# ============================================================
FACILITY_MAP_FILE = "Well_Facility_Map.csv"       # col 1 = Well Name, col 2 = Facility
PLAN_PRODUCTION_FILE = "PlanProduction.csv"
PLAN_FORECAST_FILE = "PlanForecast.csv"
MODIFIED_PRODUCTION_FILE = "ModifiedProduction.csv"
MODIFIED_FORECAST_FILE = "ModifiedForecast.csv"


def normalize_well_name(x):
    if pd.isna(x):
        return np.nan
    return re.sub(r"\s+", " ", str(x).strip()).upper()


def parse_date(s):
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp()


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x).strip()).strip("_") or "Facility"


def read_combo_csv(filename):
    raw = pd.read_csv(filename)
    if raw.shape[1] < 10:
        raise ValueError(f"{filename} must contain at least columns A-J.")
    df = raw.iloc[:, [0, 7, 8, 9]].copy()
    df.columns = ["Well_Name", "Date", "Oil_BBL", "Gas_MCF"]
    df["Match_Name"] = df["Well_Name"].apply(normalize_well_name)
    df["Date"] = parse_date(df["Date"])
    df["Oil_BBL"] = pd.to_numeric(df["Oil_BBL"], errors="coerce").fillna(0)
    df["Gas_MCF"] = pd.to_numeric(df["Gas_MCF"], errors="coerce").fillna(0)
    df = df[df["Match_Name"].notna() & df["Date"].notna()]
    return df.groupby(["Match_Name", "Date"], as_index=False)[["Oil_BBL", "Gas_MCF"]].sum()


def build_case(case_name, prod_df, fcst_df, well_map):
    profiles, qa = [], []

    for _, r in well_map.iterrows():
        well, fac, key = r["Well_Name"], r["Facility"], r["Match_Name"]
        p = prod_df[prod_df["Match_Name"] == key].sort_values("Date").copy()
        f = fcst_df[fcst_df["Match_Name"] == key].sort_values("Date").copy()

        found_p, found_f = len(p) > 0, len(f) > 0

        if found_p:
            start = p["Date"].min()
        elif found_f:
            start = f["Date"].min()
        else:
            qa.append({
                "Case": case_name, "Facility": fac, "Well_Name": well,
                "Found_Production": False, "Found_Forecast": False,
                "QA_Status": "MISSING FROM BOTH FILES"
            })
            continue

        prof = pd.DataFrame({"Date": pd.date_range(start=start, periods=PROFILE_MONTHS, freq="MS")})
        prof["Case"], prof["Facility"], prof["Well_Name"] = case_name, fac, well

        last_actual = p["Date"].max() if found_p else pd.NaT

        pp = p[["Date", "Oil_BBL", "Gas_MCF"]].rename(
            columns={"Oil_BBL": "A_Oil", "Gas_MCF": "A_Gas"}
        )
        ff = f[["Date", "Oil_BBL", "Gas_MCF"]].rename(
            columns={"Oil_BBL": "F_Oil", "Gas_MCF": "F_Gas"}
        )
        prof = prof.merge(pp, on="Date", how="left").merge(ff, on="Date", how="left")

        if found_p:
            am = prof["Date"] <= last_actual
            fm = prof["Date"] > last_actual
        else:
            am = pd.Series(False, index=prof.index)
            fm = pd.Series(True, index=prof.index)

        prof["Oil_BBL"] = 0.0
        prof["Gas_MCF"] = 0.0
        prof["Data_Source"] = ""

        prof.loc[am, "Oil_BBL"] = prof.loc[am, "A_Oil"].fillna(0)
        prof.loc[am, "Gas_MCF"] = prof.loc[am, "A_Gas"].fillna(0)
        prof.loc[am, "Data_Source"] = "Actual"

        prof.loc[fm, "Oil_BBL"] = prof.loc[fm, "F_Oil"].fillna(0)
        prof.loc[fm, "Gas_MCF"] = prof.loc[fm, "F_Gas"].fillna(0)
        prof.loc[fm, "Data_Source"] = "Forecast"

        if not found_f:
            status = "NO FORECAST FOUND"
        elif not found_p:
            status = "NO ACTUALS - FORECAST ONLY"
        else:
            req = set(prof.loc[prof["Date"] > last_actual, "Date"])
            status = "OK" if req.issubset(set(f["Date"])) else "FORECAST DOES NOT COVER FULL 50 YEARS"

        qa.append({
            "Case": case_name, "Facility": fac, "Well_Name": well,
            "Found_Production": found_p, "Found_Forecast": found_f,
            "Profile_Start": start, "Last_Actual_Month": last_actual,
            "QA_Status": status
        })
        profiles.append(prof[["Case", "Facility", "Well_Name", "Date", "Data_Source", "Oil_BBL", "Gas_MCF"]])

    if not profiles:
        raise ValueError(f"No mapped wells found for {case_name}.")
    return pd.concat(profiles, ignore_index=True), pd.DataFrame(qa)


def aggregate_facilities(well_level):
    x = (well_level.groupby(["Facility", "Date"], as_index=False)[["Oil_BBL", "Gas_MCF"]]
         .sum().sort_values(["Facility", "Date"]))
    x["Days_In_Month"] = x["Date"].dt.days_in_month
    x["Oil_BPD"] = x["Oil_BBL"] / x["Days_In_Month"]
    x["Gas_MCFD"] = x["Gas_MCF"] / x["Days_In_Month"]
    return x


def compare_facilities(mod, plan):
    c = pd.merge(mod, plan, on=["Facility", "Date"], how="outer",
                 suffixes=("_Modified", "_Plan")).sort_values(["Facility", "Date"]).reset_index(drop=True)

    cols = [
        "Oil_BBL_Modified", "Gas_MCF_Modified", "Oil_BPD_Modified", "Gas_MCFD_Modified",
        "Oil_BBL_Plan", "Gas_MCF_Plan", "Oil_BPD_Plan", "Gas_MCFD_Plan"
    ]
    c[cols] = c[cols].fillna(0)

    c["Cum_Oil_BBL_Modified"] = c.groupby("Facility")["Oil_BBL_Modified"].cumsum()
    c["Cum_Oil_BBL_Plan"] = c.groupby("Facility")["Oil_BBL_Plan"].cumsum()
    c["Cum_Gas_MCF_Modified"] = c.groupby("Facility")["Gas_MCF_Modified"].cumsum()
    c["Cum_Gas_MCF_Plan"] = c.groupby("Facility")["Gas_MCF_Plan"].cumsum()
    return c


def plot_facility(df, facility, stream):
    d = df[df["Facility"] == facility].sort_values("Date").copy()

    if stream == "Oil":
        rmod, rplan = "Oil_BPD_Modified", "Oil_BPD_Plan"
        cmod, cplan = "Cum_Oil_BBL_Modified", "Cum_Oil_BBL_Plan"
        runit, cunit = "BPD", "BBL"
    else:
        rmod, rplan = "Gas_MCFD_Modified", "Gas_MCFD_Plan"
        cmod, cplan = "Cum_Gas_MCF_Modified", "Cum_Gas_MCF_Plan"
        runit, cunit = "MCF/D", "MCF"

    final_mod, final_plan = d[cmod].iloc[-1], d[cplan].iloc[-1]
    delta = final_mod - final_plan
    delta_pct = delta / final_plan * 100 if final_plan != 0 else np.nan
    peak_mod, peak_plan = d[rmod].max(), d[rplan].max()
    peak_overall = max(peak_mod, peak_plan)

    fig, ax1 = plt.subplots(figsize=(14, 7))
    ax2 = ax1.twinx()

    l1, = ax1.plot(d["Date"], d[rmod], linewidth=2, label="Modified Rate")
    l2, = ax1.plot(d["Date"], d[rplan], linewidth=2, label="Plan Rate")
    l3, = ax2.plot(d["Date"], d[cmod], "--", linewidth=2, label="Modified Cumulative")
    l4, = ax2.plot(d["Date"], d[cplan], "--", linewidth=2, label="Plan Cumulative")

    ax2.fill_between(d["Date"], d[cmod], d[cplan], color="red", alpha=0.18)

    ax1.set_title(f"{facility} - {stream} Rate + Cumulative: Modified vs Plan")
    ax1.set_xlabel("Date")
    ax1.set_ylabel(f"{stream} Rate ({runit})")
    ax2.set_ylabel(f"Cumulative {stream} ({cunit})")
    ax1.grid(True, alpha=0.3)

    ax2.annotate(
        f"End Cum Delta: {delta:+,.0f} {cunit}\nDelta vs Plan: {delta_pct:+.2f}%",
        xy=(d["Date"].iloc[-1], max(final_mod, final_plan)),
        xytext=(-190, -25), textcoords="offset points",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.88),
        arrowprops=dict(arrowstyle="->")
    )

    ax1.text(
        0.015, 0.97,
        f"Peak Modified: {peak_mod:,.0f} {runit}\n"
        f"Peak Plan: {peak_plan:,.0f} {runit}\n"
        f"Peak Facility Flow: {peak_overall:,.0f} {runit}",
        transform=ax1.transAxes, va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.88)
    )

    ax1.legend([l1, l2, l3, l4],
               ["Modified Rate", "Plan Rate", "Modified Cumulative", "Plan Cumulative"],
               loc="best")
    fig.tight_layout()

    if SAVE_PLOTS:
        fig.savefig(f"{safe_filename(facility)}_{stream}_Modified_vs_Plan.png",
                    dpi=160, bbox_inches="tight")
    plt.show()

    return {
        "Facility": facility,
        "Stream": stream,
        "Peak_Modified_Rate": peak_mod,
        "Peak_Plan_Rate": peak_plan,
        "Peak_Facility_Flow": peak_overall,
        "Rate_Units": runit,
        "Final_Modified_Cumulative": final_mod,
        "Final_Plan_Cumulative": final_plan,
        "Final_Cumulative_Delta": delta,
        "Final_Delta_Percent_vs_Plan": delta_pct,
        "Cumulative_Units": cunit
    }


# ============================================================
# LOAD FACILITY MAP
# ============================================================
raw_map = pd.read_csv(FACILITY_MAP_FILE)
if raw_map.shape[1] < 2:
    raise ValueError("Facility map must have at least two columns.")

well_map = raw_map.iloc[:, [0, 1]].copy()
well_map.columns = ["Well_Name", "Facility"]
well_map["Well_Name"] = well_map["Well_Name"].astype(str).str.strip()
well_map["Facility"] = well_map["Facility"].astype(str).str.strip()
well_map = well_map[
    (well_map["Well_Name"] != "") &
    (well_map["Facility"] != "") &
    (well_map["Well_Name"].str.lower() != "nan") &
    (well_map["Facility"].str.lower() != "nan")
].copy()
well_map["Match_Name"] = well_map["Well_Name"].apply(normalize_well_name)

dup_fac = well_map.groupby("Match_Name")["Facility"].nunique()
bad = dup_fac[dup_fac > 1]
if len(bad):
    raise ValueError("A well is assigned to multiple facilities: " + ", ".join(bad.index[:10]))

well_map = well_map.drop_duplicates("Match_Name").reset_index(drop=True)

print("Mapped wells:", len(well_map))
print("Facilities:", well_map["Facility"].nunique())


# ============================================================
# LOAD FOUR LARGE CSVs
# ============================================================
plan_prod = read_combo_csv(PLAN_PRODUCTION_FILE)
plan_fcst = read_combo_csv(PLAN_FORECAST_FILE)
mod_prod = read_combo_csv(MODIFIED_PRODUCTION_FILE)
mod_fcst = read_combo_csv(MODIFIED_FORECAST_FILE)


# ============================================================
# BUILD CASES
# ============================================================
modified_well_level, modified_qa = build_case(
    "Modified Profile", mod_prod, mod_fcst, well_map
)
plan_well_level, plan_qa = build_case(
    "Plan Profile", plan_prod, plan_fcst, well_map
)

print("\nModified QA:")
print(modified_qa["QA_Status"].value_counts(dropna=False))
print("\nPlan QA:")
print(plan_qa["QA_Status"].value_counts(dropna=False))


# ============================================================
# FACILITY AGGREGATION AND COMPARISON
# ============================================================
modified_facility = aggregate_facilities(modified_well_level)
plan_facility = aggregate_facilities(plan_well_level)
facility_comparison = compare_facilities(modified_facility, plan_facility)


# ============================================================
# PLOT EVERY FACILITY: ONE OIL CHART + ONE GAS CHART
# ============================================================
summary_rows = []
for facility in sorted(facility_comparison["Facility"].dropna().unique()):
    print("\n" + "=" * 80)
    print(facility)
    print("=" * 80)
    summary_rows.append(plot_facility(facility_comparison, facility, "Oil"))
    summary_rows.append(plot_facility(facility_comparison, facility, "Gas"))

facility_summary = pd.DataFrame(summary_rows)
display(facility_summary)


# ============================================================
# COMPACT ONE-ROW-PER-FACILITY SUMMARY
# ============================================================
oil = facility_summary[facility_summary["Stream"] == "Oil"].copy()
gas = facility_summary[facility_summary["Stream"] == "Gas"].copy()

oil = oil.rename(columns={
    "Peak_Modified_Rate": "Peak_Modified_Oil_BPD",
    "Peak_Plan_Rate": "Peak_Plan_Oil_BPD",
    "Peak_Facility_Flow": "Peak_Oil_BPD",
    "Final_Modified_Cumulative": "Final_Modified_Oil_BBL",
    "Final_Plan_Cumulative": "Final_Plan_Oil_BBL",
    "Final_Cumulative_Delta": "Final_Oil_Delta_BBL",
    "Final_Delta_Percent_vs_Plan": "Final_Oil_Delta_Pct_vs_Plan"
})
gas = gas.rename(columns={
    "Peak_Modified_Rate": "Peak_Modified_Gas_MCFD",
    "Peak_Plan_Rate": "Peak_Plan_Gas_MCFD",
    "Peak_Facility_Flow": "Peak_Gas_MCFD",
    "Final_Modified_Cumulative": "Final_Modified_Gas_MCF",
    "Final_Plan_Cumulative": "Final_Plan_Gas_MCF",
    "Final_Cumulative_Delta": "Final_Gas_Delta_MCF",
    "Final_Delta_Percent_vs_Plan": "Final_Gas_Delta_Pct_vs_Plan"
})

facility_peak_summary = pd.merge(
    oil[[
        "Facility", "Peak_Modified_Oil_BPD", "Peak_Plan_Oil_BPD", "Peak_Oil_BPD",
        "Final_Modified_Oil_BBL", "Final_Plan_Oil_BBL",
        "Final_Oil_Delta_BBL", "Final_Oil_Delta_Pct_vs_Plan"
    ]],
    gas[[
        "Facility", "Peak_Modified_Gas_MCFD", "Peak_Plan_Gas_MCFD", "Peak_Gas_MCFD",
        "Final_Modified_Gas_MCF", "Final_Plan_Gas_MCF",
        "Final_Gas_Delta_MCF", "Final_Gas_Delta_Pct_vs_Plan"
    ]],
    on="Facility", how="outer"
)

display(facility_peak_summary)


# ============================================================
# SAVE CSV OUTPUTS
# ============================================================
modified_qa.to_csv("Modified_QA.csv", index=False)
plan_qa.to_csv("Plan_QA.csv", index=False)
modified_facility.to_csv("Modified_Facility_Aggregated_50yr.csv", index=False)
plan_facility.to_csv("Plan_Facility_Aggregated_50yr.csv", index=False)
facility_comparison.to_csv("Facility_Modified_vs_Plan_Comparison.csv", index=False)
facility_peak_summary.to_csv("Facility_Peak_and_Cumulative_Delta_Summary.csv", index=False)

print("\nComplete.")
print("Each facility has an Oil PNG and Gas PNG if SAVE_PLOTS = True.")
